# Выдал метрику 384...
## Валидация: 43.34
Отобрал сильно корелированые фичи на основе хитмапа. Их всего 3, улучшило метрику не сильно где то на 0.02
Добавили фичей на основе этих трёх, дало прирост 8, неплохо
Перебрал гиперпараметр и использовал кросс валидацию, что дало минимальный прирост около 0.01
### Вывод: линейная модель как не крути хуже чем более актуальные модели,типа деревьев

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor
from src.const import PATH_TO_DATA, RANDOM_STATE
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [13]:
from src.utils import load_data
import pygame
df_x_train, df_y_train, df_test = load_data('../data_pizdata')

In [14]:
dublicates = [(4, 12), (5, 13), (6, 14), (7, 15)]
try:
    print('Дубликаты: ')
    for i in dublicates:
        print(f'{i}: {np.sum(df_x_train.iloc[:, i[0]] == df_x_train.iloc[:, i[1]])} из {len(df_x_train.iloc[:, 0])}')
        
    df_x_train.drop(columns=[i[0] for i in dublicates], inplace=True)
    df_test.drop(columns=[i[0] for i in dublicates], inplace=True)
    print('Дубликаты удалены')
    
except Exception:
    print('Дубликаты уже удалены')

print(df_x_train.shape[1])
assert df_x_train.shape[1] == 12

Дубликаты: 
(4, 12): 40000 из 40000
(5, 13): 40000 из 40000
(6, 14): 40000 из 40000
(7, 15): 40000 из 40000
Дубликаты удалены
12


In [15]:
def add_features(X):
    # Старые фичи (27, 28, 39, 41, 42)
    new_features = []
    
    # Фича 27: сумма по строкам
    sum_f = np.sum(X, axis=1, keepdims=True)
    new_features.append(sum_f)
    
    # Фича 28: среднее по строкам
    mean_f = np.mean(X, axis=1, keepdims=True)
    new_features.append(mean_f)
    
    # Фича 39: количество значений выше среднего
    count_above_mean = np.sum(X > mean_f, axis=1, keepdims=True)
    new_features.append(count_above_mean)
    
    # Фича 41: логарифм абсолютных значений для первых 4 признаков
    log_features = np.log1p(np.abs(X[:, :4]))
    new_features.append(log_features)
    
    # Фича 42: квадратный корень для признаков 4-8
    sqrt_features = np.sqrt(np.abs(X[:, 4:8]) + 1e-6)
    new_features.append(sqrt_features)

    # Новые преобразования для фич 10, 11, 12 (индексы 9,10,11 в X)
    f10 = X[:, 9:10]
    f11 = X[:, 10:11]
    f12 = X[:, 11:12]

    # 1. Нелинейные преобразования для f12 (корреляция -0.86)
    new_features.append(np.sqrt(np.abs(f12)))          # Фича 43
    new_features.append(f12 ** 2)                      # Фича 44

    # 2. Взаимодействие фич 10-11-12
    new_features.append(f10 * f12)                     # Фича 45
    new_features.append((f11 + f12) / (f10 + 1e-6))    # Фича 46

    # 3. Условные преобразования
    new_features.append(np.where(f12 < np.median(f12), 0, 1)) # Фича 47
    new_features.append(np.log1p(np.abs(f10 + f11)))   # Фича 48

    # 4. Агрегаты
    new_features.append((f10 + f11 + f12) / 3)         # Фича 49
    new_features.append(np.max([f10, f11, f12], axis=0)) # Фича 50

    return np.hstack([X] + new_features)


df_x_train = add_features(df_x_train.values)
df_test = add_features(df_test.values)

In [ ]:
df_train = df_x_train.copy()
df_train = np.hstack([df_train, df_y_train])
corr_matrix = pd.DataFrame(df_train).corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Тепловая карта корреляций")
plt.show()

In [6]:
df_x_train.drop(columns=[0, 1, 2, 3, 4, 5, 6, 7, 8], inplace=True)
df_x_train.drop(columns=[0, 1, 2, 3, 4, 5, 6, 7, 8], inplace=True)

array([[0.4510095 , 0.61460638, 0.16730614, ..., 2.27122476, 5.16113601,
        6.79214501],
       [0.8371234 , 0.55844869, 0.03419815, ..., 1.31779557, 1.01999645,
        2.        ],
       [0.60563576, 0.35095504, 0.67840821, ..., 1.59361829, 3.51079434,
        6.61085878],
       ...,
       [0.4811432 , 0.70772098, 0.88116348, ..., 1.87793471, 2.16079527,
        3.53998395],
       [0.69956601, 0.90805913, 0.57516987, ..., 1.44387328, 1.35126554,
        2.        ],
       [0.60549189, 0.11642799, 0.08077757, ..., 1.27016495, 1.21970883,
        1.56143998]])

In [7]:
X = df_x_train.copy()
y = df_y_train[0].values  # или df_y_train.squeeze() если это Series

# Разбиваем данные на обучающую и валидационную выборки
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y, 
    test_size=0.2,
    random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(random_state=42))
])

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_grid = {
    'ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Лучшие параметры:", grid_search.best_params_)
print("Лучшее отрицательное MSE (на кросс-валидации):", grid_search.best_score_)

val_pred = grid_search.predict(X_val)
mse = mean_squared_error(y_val, val_pred)
print(f"MSE на валидации: {mse:.4f}")

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Лучшие параметры: {'ridge__alpha': 10.0}
Лучшее отрицательное MSE (на кросс-валидации): -47.368162474145116
MSE на валидации: 48.4030


In [28]:
# model = Ridge(random_state=RANDOM_STATE, alpha=0.1)
# model.fit(X_train, y_train)
# 
# val_pred = model.predict(X_val)
# mse = mean_squared_error(y_val, val_pred)
# print(f"MSE на валидации: {mse:.4f}")

MSE на валидации: 43.3486


MSE на валидации: 48.37 оставлены все фичи в том числе не коррелированые
MSE на валидации: 48.3595 без добавления фич  
MSE на валидации: 43.3486 с добавлением фич
MSE на валидации: 43.3474 с кросс валидацией и перебором параметров

In [36]:
from src.utils import save_predictions

y_pred = grid_search.predict(df_test)
save_predictions(y_pred)

Предсказания сохранены в: predictions\predictions_7.csv


'predictions\\predictions_7.csv'